In [ ]:
import OpenAI from "npm:openai"
import { ChatOpenAI } from "npm:@langchain/openai"
import { StateGraph, END } from "npm:@langchain/langgraph"
import { z } from "npm:zod"
import { zodToJsonSchema } from "npm:zod-to-json-schema"
import { load } from "jsr:@std/dotenv"
const env = await load()
const apiKey = env["OPENAI_API_KEY"]

#### Create our tools

Create two fake tools: `search` and `calculator`.
Each needs:
* The function to be executed
* A tool definition with:
    * the tool name
    * the tool decription
    * the arguments schema

Wrap the tools in a LangGraph `ToolNode`. It will automatically handle for us the tool execution based on the LLM tool calls and append tools results in the State `message`key.

In [ ]:
import { ToolNode } from "@langchain/langgraph/prebuilt";
import { tool } from "@langchain/core/tools";

const search = tool(
  ({ query }) => `Results for: ${query}`,
  {
    name: "search",
    description: "Search for information.",
    schema: z.object({ query: z.string() }),
  }
);

const calculator = tool(
  ({ expression }) => String(eval(expression)),
  {
    name: "calculator",
    description: "Evaluate a math expression.",
    schema: z.object({ expression: z.string() }),
  }
);

const toolNode = new ToolNode([search, calculator]);

#### The Agent state

In [ ]:
import * as z from "zod";
import {
  StateGraph,
  StateSchema,
  MessagesValue,
  START,
  END,
  type GraphNode,
  type ConditionalEdgeRouter,
} from "@langchain/langgraph";

const AgentState = new StateSchema({
    messages: MessagesValue,              // Will hold the conversation history
    iterations: z.number().default(0),   // Count of how many times we've looped through the LLM
    max_iterations: z.number().default(5), // Max iterations before we force a tool call
});

#### The Agent nodes

In [ ]:
import * as z from "zod";
import {
  StateGraph,
  StateSchema,
  MessagesValue,
  START,
  END,
  type GraphNode,
  type ConditionalEdgeRouter,
} from "@langchain/langgraph";

import { PromptTemplate, 
  SystemMessagePromptTemplate, 
  MessagesPlaceholder, 
  HumanMessagePromptTemplate }
    from "langchain";

// Graph API: Clear visualization of decision paths

const routeToolCalls: ConditionalEdgeRouter<typeof AgentState> = (state) => {
    const lastMessage = state.messages[state.messages.length - 1];
    return  ( 
      state.iterations < state.max_iterations &&  // Check if we haven't exceeded max iterations
      lastMessage.tool_calls                      // Check if the last message included any tool calls
      && lastMessage.tool_calls.length > 0 )      // Ensure there was at least one tool call
        ? "toolCalled" :  // If conditions are met, route to "toolCalled"
        "noTool";       // If conditions are not met, route to "noTool"

};

const callLlm: GraphNode<typeof AgentState> = (state) => {
  const llm = new ChatOpenAI({ model: modelName });
  const promptTemplate = PromptTemplate.fromMessages([
    SystemMessagePromptTemplate.fromTemplate("You are an assistant that can call tools."),
    MessagesPlaceholder.fromName("messages"),
    HumanMessagePromptTemplate.fromTemplate("Based on the conversation, decide if you need to call a tool. Respond with 'Yes' or 'No'."),
  ]);
  return {
    ...state,
    currentTool: "search", // Example: decide to call search tool
  };
}

const searchNode: GraphNode<typeof AgentState> = (state) => {
  // Call search tool and update state
  return {
    ...state,
    retryCount: state.retryCount + 1, // Increment retry count
  };
}

const workflow = new StateGraph(AgentState)
  .addNode("callLlm", callLlmNode)
  .addNode("processSearch", searchNode)
  .addConditionalEdges("callLlm", shouldContinue);